In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
# Load dataset
df = pd.read_csv("spam.csv", encoding='latin-1')

df.head()

In [ ]:
# Data Cleaning
# Remove unnecessary columns
df = df[['v1','v2']]

In [ ]:
# Rename columns
df.columns = ['label', 'message']

In [ ]:
# Check missing values
print(df.isnull().sum())

In [ ]:
# Check duplicates
print(df.duplicated().sum())

df.drop_duplicates(inplace=True)

In [ ]:
# Text Preprocessing
nltk.download('stopwords')

ps = PorterStemmer()

def transform_text(text):
    text = text.lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [ps.stem(word)
             for word in words
             if word not in stopwords.words('english')]

    return " ".join(words)

df['transformed_message'] = df['message'].apply(transform_text)

In [ ]:
# Convert Text to Numbers using TF-IDF
tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['transformed_message']).toarray()

y = df['label']

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Train Model
model = MultinomialNB()

model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = model.predict(X_test)

In [ ]:
#Model Evaluation
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label='spam'))
print("Recall   :", recall_score(y_test, y_pred, pos_label='spam'))
print("F1 Score :", f1_score(y_test, y_pred, pos_label='spam'))

In [ ]:
#Classification Report
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# User Input Prediction
msg = input("Enter Message: ")

msg = transform_text(msg)

vector_input = tfidf.transform([msg]).toarray()

result = model.predict(vector_input)

print("Prediction:", result[0])